In [0]:
bronze_df= spark.read.format("delta")\
    .load("/Volumes/workspace/bronzebike/bronzevolume/bronzedata/data")

display(bronze_df)    

In [0]:
# --- Notebook: 02_Refine_Silver ---

In [0]:
from pyspark.sql.functions import col, unix_timestamp, round, expr, when, to_timestamp



# 1. Cast coordinates to Double (Numbers)
bronze_df = bronze_df.withColumn("start_lat", col("start_lat").cast("double")) \
                     .withColumn("start_lng", col("start_lng").cast("double")) \
                     .withColumn("end_lat", col("end_lat").cast("double")) \
                     .withColumn("end_lng", col("end_lng").cast("double"))


bronze_df = bronze_df.withColumn("started_at", 
    to_timestamp(col("started_at"), "yyyy-MM-dd HH:mm:ss.SSS")
).withColumn("ended_at", 
    to_timestamp(col("ended_at"), "yyyy-MM-dd HH:mm:ss.SSS")
)                     

haversine_expr = """
    2 * 6371 * asin(sqrt(
        pow(sin(radians(end_lat - start_lat) / 2), 2) +
        cos(radians(start_lat)) * cos(radians(end_lat)) *
        pow(sin(radians(end_lng - start_lng) / 2), 2)
    ))
"""


silver_df = (bronze_df
    # 1. Clean NULLs and "False Starts"
    .filter(col("end_lat").isNotNull() & (col("start_station_id") != col("end_station_id")))
    .withColumn("duration_sec", unix_timestamp(col("ended_at")) - unix_timestamp(col("started_at")))
    .filter(col("duration_sec").between(60, 86400))

    # 2. Standardize Membership
    .withColumn("is_member", col("member_casual") == "member")

    # 3. Categorize Bike Type
    .withColumn("bike_category", 
        when(col("rideable_type").contains("electric"), "Electric")
        .otherwise("Manual"))

    # 4. Geospatial & Speed

    .withColumn("distance_km", round(expr(haversine_expr), 2))
    .withColumn("avg_speed_kmh", 
        when(col("duration_sec") > 0, 
             round(col("distance_km") / (col("duration_sec") / 3600), 2))
        .otherwise(0))


 )

silver_df=silver_df.drop("_rescued_data")
display(silver_df)

In [0]:
# --- Silver Layer: silver_trips (The "Fact" Stream) ---

silver_fact_df=(silver_df
    .withColumn("is_member", col("member_casual") == "member")
    .withColumn("bike_category", when(col("rideable_type").contains("electric"), "Electric").otherwise("Manual"))
    
    # D. DROP Descriptive Columns (They belong in the Dimension Table)
    .drop("start_station_name", "end_station_name", "member_casual", "rideable_type"))

# 2. Write to Silver Fact Table
silver_fact_df.write.format("delta").mode("overwrite").save("/Volumes/workspace/silverbike/silvervolume/silverdata/silver_trips")
## Why this split is correct:


In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, current_timestamp, lit, to_timestamp

# --- 1. CONFIGURATION ---
# Replace this with your actual Volume path
target_path = "/Volumes/workspace/silverbike/silvervolume/silverdata/dim_stations"

# --- 2. TRANSFORMATION LOGIC ---
# This function prepares your data with the required SCD2 columns
def prepare_stations(df):
    return (df.select("start_station_id", "start_station_name", "start_lat", "start_lng")
            .distinct()
            .withColumnRenamed("start_station_id", "station_id")
            .withColumnRenamed("start_station_name", "station_name")
            .withColumnRenamed("start_lat", "lat")
            .withColumnRenamed("start_lng", "lng")
            .withColumn("start_date", current_timestamp())
            .withColumn("end_date", to_timestamp(lit("9999-12-31")))
            .withColumn("is_current", lit(True))
           )

# Assigning your data (assuming silver_base_df is your cleaned trip data)
# On Day 1, initial_df and source_df are effectively created the same way
source_df = prepare_stations(silver_df)

# --- 3. INITIALIZATION (The "Cold Start") ---
# If the folder doesn't exist, we "Bootstrap" it by saving the data directly
if not DeltaTable.isDeltaTable(spark, target_path):
    print(f"Initializing target folder at: {target_path}")
    source_df.write.format("delta").mode("overwrite").save(target_path)
    print("Bootstrap complete. Table created.")

# --- 4. INCREMENTAL SCD2 MERGE ---
else:
    print("Target exists. Running Incremental SCD2 Merge...")
    target_table = DeltaTable.forPath(spark, target_path)

    target_table.alias("target").merge(
        source_df.alias("source"),
        "target.station_id = source.station_id AND target.is_current = true"
    ).whenMatchedUpdate(
        # If name or coordinates changed, "expire" the current record
        condition="""
            target.station_name <> source.station_name OR 
            target.lat <> source.lat OR 
            target.lng <> source.lng
        """,
        set={
            "is_current": lit(False),
            "end_date": current_timestamp()
        }
    ).whenNotMatchedInsert(
        # If it's a brand new station ID, insert it
        values={
            "station_id": col("source.station_id"),
            "station_name": col("source.station_name"),
            "lat": col("source.lat"),
            "lng": col("source.lng"),
            "start_date": current_timestamp(),
            "end_date": to_timestamp(lit("9999-12-31")),
            "is_current": lit(True)
        }
    ).execute()
    print("Merge complete.")

# --- 5. VERIFICATION ---
display(spark.read.format("delta").load(target_path).sort("station_id", "start_date"))

In [0]:
%skip
from pyspark.sql.functions import col, current_timestamp, lit, to_timestamp
# Create the initial list of stations from your cleaned data
initial_df = (silver_df
              .select("start_station_id", "start_station_name", "start_lat", "start_lng")
              .distinct()
              .withColumnRenamed("start_station_id", "station_id")
              .withColumnRenamed("start_station_name", "station_name")
              .withColumnRenamed("start_lat", "lat")
              .withColumnRenamed("start_lng", "lng")
              # Add the SCD2 columns for the first time
              .withColumn("start_date", current_timestamp())
              .withColumn("end_date", to_timestamp(lit("9999-12-31")))
              .withColumn("is_current", lit(True))
              )

In [0]:
%skip
(initial_df.write
  .format("delta")
  .mode("overwrite")
  .save("/Volumes/workspace/silverbike/silvervolume/silverdata/dim_stations"))

In [0]:
%skip
source_df = (silver_df
             .select("start_station_id", "start_station_name", "start_lat", "start_lng")
             .distinct()
             .withColumnRenamed("start_station_id", "station_id")
             .withColumnRenamed("start_station_name", "station_name")
             .withColumnRenamed("start_lat", "lat")
             .withColumnRenamed("start_lng", "lng")
            )

In [0]:
%skip
from delta.tables import DeltaTable
from pyspark.sql.functions import col, current_timestamp, lit, to_timestamp

target_path = "/Volumes/workspace/silverbike/silvervolume/silverdata/dim_stations"
target_table = DeltaTable.forPath(spark, target_path)
# Execute the SCD2 logic
target_table.alias("target").merge(
    source_df.alias("source"),
    "target.station_id = source.station_id AND target.is_current = true"
).whenMatchedUpdate(
    condition="target.station_name <> source.station_name",
    set={
        "is_current": lit(False), 
        "end_date": current_timestamp()
    }
).whenNotMatchedInsert(
    values={
        "station_id": col("source.station_id"),
        "station_name": col("source.station_name"), # Fixed column name here
        "start_date": current_timestamp(),
        "end_date": to_timestamp(lit("9999-12-31")),
        "is_current": lit(True)
    }
).execute()